# Eval Overview - Gene Level Analysis

Our second eval category is reviews our gene expression prediction from the perspective of the gene dimension. This eval runs on top of the predictions produced by the [linear expression decoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_decoders.ipynb). This analysis contines from the [Gene Expression Predictions](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_expr_prediction.ipynb) focusing on analyzing perturbation level expression predictions from the gene axis.  


`<TO BE UPDATED>`

Our eval suite does 3 levels of analysis on the expression prediction: sample-level, per-perturbation, and cross-perturbation. We perform these on the full test set, on test sets subset by dataset, and on test sets subset by cell type. As you walk through the notebook you'll see that we evaluate on multiple dimensions to ensure we have a thorough understanding on where our model is working well and where it's struggling.

In [1]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import torch

In [2]:
SEED = 1337
np.random.seed(SEED)
torch.manual_seed(SEED)

## Data Prep
We'll start by preparing our data. For this evaluation, we need the average change in expression *expression delta* by perturbation.  

To get this we need to have the true cell expressions, the control expression for each cell, and the perturbation information. A "perturbation" is a unique combination of a sequence, target, modality, and mode applied to a cell type. A perturbation can span datasets. For the 6 samples we'll show the real control cell expression, predicted control cell expression (running the student encoder and then linear decoder), real case cell expression, predicted case cell expression, and the perturbation. 

For predicted expression, we use the data created by the [linear expression decoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_decoders.ipynb). As a reminder, the decoder takes the mean ($\mu$) BioJEPA-AC output based on the perturbations and cell expression pattern, and then uses a linear layer to project down to a $[\text{n\_genes},\text{1}]$ matrix with a single value per gene representing the expression.

After we stage the expression counts by sample, we'll calcluate the expression delta by perturbation. This requires grouping our sample data by perturbation and taking the mean. With the staged data, you'll see that the first two samples share a perturbation as do our third and fourth sample.  

We'll stage data to show a few different predictions: a strong prediction, weak prediction, inverse prediction, and over prediction.

In [3]:
unique_perts = 4
num_genes = 8
num_cells = 6
TOP_K = 3

**Perturbations**

We'll first start with our perturbations. Even though we have 6 cells, our sample will only have 4 unique perturbations. To define a unique perturbation, it's not just about what we target, but the context of it. Because of this, we represent a unique perturbation as $\text{(seq id, targ id, modality id, mode id, cell type)}$. IDs are used since our model keeps the perturbation information in separate caches from our sample expression counts to avoid heavy duplication of information.

We'll also create a mapping of the sample to the perturbation, where the first two samples map to the first perturbation, the next two samples to the second perturbation, and then the final two samples each have a unique perturbation.

In [4]:
pert_keys = [                                                                      
      (0, 0, 0, 0, 0),  # pert 0: DNA CRISPRi, cell type 0 
      (1, 1, 0, 0, 0),  # pert 1: DNA CRISPRi, cell type 0 
      (2, 2, 0, 1, 0),  # pert 2: DNA CRISPRa, cell type 0 
      (3, 3, 2, 4, 0),  # pert 3: Chemical inhibitor, cell type 0
]


sample_to_pert = [0, 0, 1, 1, 2, 3]

**Control Cells**

Next we'll show the control cell values. Recall that for our inference we pair together a perturbed cell with a random control cell from the same batch. This allows us to take an approximate change in prediction. Beyond just the control cell expression, to calculate our predicted change in expression, we run the expression predictor on the control cell's latent representation `z_context`. We'll discuss the calculation more but to show this we'll also have the predicted control expression.

For the predicted, we'll show a minor shift to highlight that often the prediction is not perfect.

In [5]:
real_control = np.array([
    [2.1, 3.4, 1.2, 4.1, 2.6, 3.1, 1.4, 4.6], 
    [1.9, 3.6, 0.8, 3.9, 2.4, 2.9, 1.6, 4.4], 
    [2.0, 3.3, 1.1, 4.2, 2.3, 3.2, 1.3, 4.3], 
    [2.2, 3.7, 0.9, 3.8, 2.7, 2.8, 1.7, 4.7], 
    [2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 4.5], 
    [2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 4.5], 
])
pred_control = np.array([
    [2.0, 3.3, 1.3, 4.0, 2.5, 3.2, 1.3, 4.5], 
    [1.8, 3.5, 0.9, 3.8, 2.3, 3.0, 1.5, 4.3], 
    [1.9, 3.2, 1.2, 4.1, 2.2, 3.3, 1.2, 4.2], 
    [2.1, 3.6, 1.0, 3.7, 2.6, 2.9, 1.6, 4.6], 
    [1.9, 3.4, 1.1, 3.9, 2.4, 3.1, 1.4, 4.4], 
    [1.9, 3.4, 1.1, 3.9, 2.4, 3.1, 1.4, 4.4], 
])
real_control.shape, pred_control.shape

((6, 8), (6, 8))

**Case Cell**

Next we'll show the case cell values. In our raw data we have the real expression of the perturbed cell. We pair this together with the linear expression decoder output based on the mean prediction, $\mu$, from the ACPredictor output. The mean prediction is based on the control cell latent representation `z_context` and the perturbations. You'll quickly be able to see that there is a difference between the real values and the predicted values. We've staged the data so that the first two samples predict closely, the next two samples weakly, the fifth sample predicts in the wrong direction, and the final sample over predicts. You'll see how these calculations flow through.

In [6]:
real_case = np.array([
    [2.8, 2.3, 1.6, 6.0, 2.0, 4.7, 1.2, 2.9], 
    [2.8, 2.2, 1.0, 6.0, 2.0, 4.3, 1.6, 2.5], 
    [2.1, 3.2, 1.2, 4.1, 2.4, 3.3, 1.3, 4.4], 
    [2.3, 3.6, 0.9, 3.7, 2.7, 2.8, 1.7, 4.7], 
    [2.5, 2.7, 1.6, 3.0, 2.8, 3.7, 1.1, 5.7], 
    [2.3, 3.1, 1.2, 4.5, 2.2, 3.6, 1.4, 4.9], 
])

pred_case = np.array([
    [2.6, 2.3, 1.8, 5.8, 2.0, 4.6, 1.0, 3.0],  # strong
    [2.6, 2.3, 1.2, 5.8, 2.0, 4.5, 1.4, 2.6],  # strong
    [2.05, 3.05, 1.25, 4.05, 2.3, 3.35, 1.15, 4.3],  # weak
    [2.15, 3.6, 0.95, 3.6, 2.65, 2.95, 1.6, 4.65],  # weak
    [1.6, 3.9, 1.6, 4.5, 2.2, 3.9, 1.7, 3.9],  # inverse
    [2.8, 2.2, 1.7, 5.4, 1.5, 4.9, 1.1, 5.6],  # over
])

real_case.shape, pred_case.shape

((6, 8), (6, 8))

**Calculate Sample Delta**

A major component of our expression benchmark is not looking at absolute predictions, but the change in expression. Some claim that this simplifies the task. Biologically, we see this as addressing the important questions: can you predict what will change, in what direction, and by how much. We focus on calculating two sample level differences:
1. `pred_delta` - the predicted change in expression as calculated by $\hat{\delta}_g = \hat{x}^{\text{case}}_g - \hat{x}^{\text{ctrl}}_g$. This value compares the predicted perturbed expression (`pred_case`) from the predicted control expression (`pred_control`). We use the predicted control expression to isolate BioJEPA-AC's learned perturbation effect from any baseline reconstruction error.
2. `real_delta` - the real change in expression as calculated by $\delta_g = x^{\text{case}}_g - x^{\text{ctrl}}_g$. This is our source of truth.

With this calculation you'll see how we end up seeing both increases and decreases in expression by gene. We'll end up comparing these by different slices in our calculations.

In [7]:
pred_delta = pred_case - pred_control

pred_delta.shape, pred_delta

((6, 8),
 array([[ 0.6 , -1.  ,  0.5 ,  1.8 , -0.5 ,  1.4 , -0.3 , -1.5 ],
        [ 0.8 , -1.2 ,  0.3 ,  2.  , -0.3 ,  1.5 , -0.1 , -1.7 ],
        [ 0.15, -0.15,  0.05, -0.05,  0.1 ,  0.05, -0.05,  0.1 ],
        [ 0.05,  0.  , -0.05, -0.1 ,  0.05,  0.05,  0.  ,  0.05],
        [-0.3 ,  0.5 ,  0.5 ,  0.6 , -0.2 ,  0.8 ,  0.3 , -0.5 ],
        [ 0.9 , -1.2 ,  0.6 ,  1.5 , -0.9 ,  1.8 , -0.3 ,  1.2 ]]))

In [8]:
real_delta = real_case - real_control

real_delta.shape, real_delta

((6, 8),
 array([[ 0.7, -1.1,  0.4,  1.9, -0.6,  1.6, -0.2, -1.7],
        [ 0.9, -1.4,  0.2,  2.1, -0.4,  1.4,  0. , -1.9],
        [ 0.1, -0.1,  0.1, -0.1,  0.1,  0.1,  0. ,  0.1],
        [ 0.1, -0.1,  0. , -0.1,  0. ,  0. ,  0. ,  0. ],
        [ 0.5, -0.8,  0.6, -1. ,  0.3,  0.7, -0.4,  1.2],
        [ 0.3, -0.4,  0.2,  0.5, -0.3,  0.6, -0.1,  0.4]]))

**Calculate Per-Perturbation Delta**

Now we'll calculate our per-perturbation data. We'll use our `sample_to_pert` to help identify which perturbation each sample belongs to. We'll iterate through our perturbations, find which samples belong to the perturbation, pluck out the expression data for the samples, and then run a mean across them to get a single value per perturbation.

In [9]:
mean_pert_pred_delta = np.zeros((unique_perts, num_genes)) 
mean_pert_real_delta = np.zeros((unique_perts, num_genes)) 

In [10]:
for pert_idx in range(unique_perts):                                            
    mask = [i for i, p in enumerate(sample_to_pert) if p == pert_idx]           
    mean_pert_pred_delta[pert_idx] = pred_delta[mask].mean(axis=0)                   
    mean_pert_real_delta[pert_idx] = real_delta[mask].mean(axis=0)
    
mean_pert_pred_delta.shape, mean_pert_pred_delta, mean_pert_real_delta

((4, 8),
 array([[ 0.7  , -1.1  ,  0.4  ,  1.9  , -0.4  ,  1.45 , -0.2  , -1.6  ],
        [ 0.1  , -0.075,  0.   , -0.075,  0.075,  0.05 , -0.025,  0.075],
        [-0.3  ,  0.5  ,  0.5  ,  0.6  , -0.2  ,  0.8  ,  0.3  , -0.5  ],
        [ 0.9  , -1.2  ,  0.6  ,  1.5  , -0.9  ,  1.8  , -0.3  ,  1.2  ]]),
 array([[ 0.8 , -1.25,  0.3 ,  2.  , -0.5 ,  1.5 , -0.1 , -1.8 ],
        [ 0.1 , -0.1 ,  0.05, -0.1 ,  0.05,  0.05,  0.  ,  0.05],
        [ 0.5 , -0.8 ,  0.6 , -1.  ,  0.3 ,  0.7 , -0.4 ,  1.2 ],
        [ 0.3 , -0.4 ,  0.2 ,  0.5 , -0.3 ,  0.6 , -0.1 ,  0.4 ]]))

## Direction Of Effect Analysis

This analsysis uses hard values to create an up/down/ or unchanged classification 

**Classify Direction All**

In [22]:
threshold = 0.25
all_pred_dir = []
all_real_dir = []

In [23]:
for p in range(unique_perts):
    print(f'--- Pert {p} ---')
    delta = mean_pert_pred_delta[p]
    direction = np.zeros_like(delta, dtype=np.int8)
    direction[delta >= threshold] = 1
    direction[delta <= -threshold] = -1
    print(direction)
    all_pred_dir.append(direction)

all_pred_dir = np.concat(all_pred_dir)
all_pred_dir.shape, all_pred_dir

--- Pert 0 ---
[ 1 -1  1  1 -1  1  0 -1]
--- Pert 1 ---
[0 0 0 0 0 0 0 0]
--- Pert 2 ---
[-1  1  1  1  0  1  1 -1]
--- Pert 3 ---
[ 1 -1  1  1 -1  1 -1  1]


((32,),
 array([ 1, -1,  1,  1, -1,  1,  0, -1,  0,  0,  0,  0,  0,  0,  0,  0, -1,
         1,  1,  1,  0,  1,  1, -1,  1, -1,  1,  1, -1,  1, -1,  1],
       dtype=int8))

In [24]:
for p in range(unique_perts):
    print(f'--- Pert {p} ---')
    delta = mean_pert_real_delta[p]
    direction = np.zeros_like(delta, dtype=np.int8)
    direction[delta >= threshold] = 1
    direction[delta <= -threshold] = -1
    print(direction)
    all_real_dir.append(direction)

all_real_dir = np.concat(all_real_dir)
all_real_dir.shape, all_real_dir

--- Pert 0 ---
[ 1 -1  1  1 -1  1  0 -1]
--- Pert 1 ---
[0 0 0 0 0 0 0 0]
--- Pert 2 ---
[ 1 -1  1 -1  1  1 -1  1]
--- Pert 3 ---
[ 1 -1  0  1 -1  1  0  1]


((32,),
 array([ 1, -1,  1,  1, -1,  1,  0, -1,  0,  0,  0,  0,  0,  0,  0,  0,  1,
        -1,  1, -1,  1,  1, -1,  1,  1, -1,  0,  1, -1,  1,  0,  1],
       dtype=int8))

**Classify Direction Top K**

In [34]:
TOP_K = 3
top_deg_pred_dir = []
top_deg_real_dir = []

In [35]:
for p in range(unique_perts):
    print(f'--- Pert {p} ---')
    top_k_idx = np.argsort(np.abs(mean_pert_real_delta[p]))[-TOP_K:]
    delta = mean_pert_pred_delta[p][top_k_idx]
    direction = np.zeros_like(delta, dtype=np.int8)
    direction[delta >= threshold] = 1
    direction[delta <= -threshold] = -1
    print(direction)
    top_deg_pred_dir.append(direction)


top_deg_pred_dir = np.concat(top_deg_pred_dir)
top_deg_pred_dir.shape, top_deg_pred_dir

--- Pert 0 ---
[ 1 -1  1]
--- Pert 1 ---
[0 0 0]
--- Pert 2 ---
[ 1  1 -1]
--- Pert 3 ---
[1 1 1]


((12,), array([ 1, -1,  1,  0,  0,  0,  1,  1, -1,  1,  1,  1], dtype=int8))

In [37]:
for p in range(unique_perts):
    print(f'--- Pert {p} ---')
    top_k_idx = np.argsort(np.abs(mean_pert_real_delta[p]))[-TOP_K:]
    delta = mean_pert_real_delta[p][top_k_idx]
    direction = np.zeros_like(delta, dtype=np.int8)
    direction[delta >= threshold] = 1
    direction[delta <= -threshold] = -1
    print(direction)
    top_deg_real_dir.append(direction)

top_deg_real_dir = np.concat(top_deg_real_dir)
top_deg_real_dir.shape, top_deg_real_dir

--- Pert 0 ---
[ 1 -1  1]
--- Pert 1 ---
[0 0 0]
--- Pert 2 ---
[-1 -1  1]
--- Pert 3 ---
[1 1 1]


((12,), array([ 1, -1,  1,  0,  0,  0, -1, -1,  1,  1,  1,  1], dtype=int8))

### Overall Accuracy

**All Genes**

In [39]:
overall_accuracy = accuracy_score(all_real_dir, all_pred_dir)
overall_accuracy

0.75

**Top DEG**

In [40]:
top_deg_accuracy = accuracy_score(top_deg_real_dir, top_deg_pred_dir)
top_deg_accuracy

0.75

### F1

**F1 Up**

In [41]:
f1_up = f1_score(all_real_dir, all_pred_dir, labels=[1], average='macro', zero_division=0)
f1_up

0.7407407407407407

**F1 Down**

In [42]:
f1_down = f1_score(all_real_dir, all_pred_dir, labels=[-1], average='macro', zero_division=0)
f1_down

0.625

**F1 Unchanged**

In [43]:
f1_unchanged = f1_score(all_real_dir, all_pred_dir, labels=[0], average='macro', zero_division=0)
f1_unchanged

0.8571428571428571

## Bin Accuracy

In [86]:
all_magnitudes = np.abs(np.concat(mean_pert_real_delta))
all_magnitudes.shape, all_magnitudes

((32,),
 array([0.8 , 1.25, 0.3 , 2.  , 0.5 , 1.5 , 0.1 , 1.8 , 0.1 , 0.1 , 0.05,
        0.1 , 0.05, 0.05, 0.  , 0.05, 0.5 , 0.8 , 0.6 , 1.  , 0.3 , 0.7 ,
        0.4 , 1.2 , 0.3 , 0.4 , 0.2 , 0.5 , 0.3 , 0.6 , 0.1 , 0.4 ]))

In [87]:
all_correct = np.array(all_real_dir == all_pred_dir)
all_correct.shape, all_correct

((32,),
 array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True, False, False,
         True, False, False,  True, False, False,  True,  True, False,
         True,  True,  True, False,  True]))

In [88]:
magnitude_bins = [0, 0.25, 0.5, 1.0, 1.5, 2.0, np.inf]
bin_labels = ['0-0.25', '0.25-0.5', '0.5-1.0', '1.0-1.5', '1.5-2.0', '2.0+']
accuracy_by_magnitude = {}

In [89]:
for i in range(len(magnitude_bins) - 1):
    print(f'Bin {i}:{magnitude_bins[i]}')
    mask = (all_magnitudes >= magnitude_bins[i]) & (all_magnitudes < magnitude_bins[i + 1])
    print(mask)
    if mask.sum() > 0:
        accuracy_by_magnitude[bin_labels[i]] = {'accuracy': float(all_correct[mask].mean()), 'count': int(mask.sum())}

Bin 0:0
[False False False False False False  True False  True  True  True  True
  True  True  True  True False False False False False False False False
 False False  True False False False  True False]
Bin 1:0.25
[False False  True False False False False False False False False False
 False False False False False False False False  True False  True False
  True  True False False  True False False  True]
Bin 2:0.5
[ True False False False  True False False False False False False False
 False False False False  True  True  True False False  True False False
 False False False  True False  True False False]
Bin 3:1.0
[False  True False False False False False False False False False False
 False False False False False False False  True False False False  True
 False False False False False False False False]
Bin 4:1.5
[False False False False False  True False  True False False False False
 False False False False False False False False False False False False
 False False False Fa

In [90]:
accuracy_by_magnitude

{'0-0.25': {'accuracy': 0.8181818181818182, 'count': 11},
 '0.25-0.5': {'accuracy': 0.7142857142857143, 'count': 7},
 '0.5-1.0': {'accuracy': 0.75, 'count': 8},
 '1.0-1.5': {'accuracy': 0.3333333333333333, 'count': 3},
 '1.5-2.0': {'accuracy': 1.0, 'count': 2},
 '2.0+': {'accuracy': 1.0, 'count': 1}}